# Compare actual EnergyPlus and RC renovation results

Both engines run the same baseline, Data window replacement and Data outer insulation with the same Gothenburg weather and climate factors. Install the integration and examples extras and EnergyPlus 24.2. Set ENERGYPLUS_EXE to your executable before starting this kernel, or provide energyplus_exe when calling run_case. A missing engine is an error, not a skipped or substituted calculation.

In [ ]:
from pathlib import Path

from actual_renovation import run_case

rc = run_case(output=Path("../outputs/engine-comparison"), engine="epsm_rc")
eplus = run_case(output=Path("../outputs/engine-comparison"), engine="energyplus")
assert rc["report"].complete and eplus["report"].complete

The numerical results may differ because RC uses a reduced building calculation and EnergyPlus uses its detailed engine. Compare energy and climate savings, keeping material factors and covered uses identical. This comparison does not determine measured-building accuracy.

In [ ]:
from teaching_case import show_table

rows = []
for step, (rc_run, ep_run) in enumerate(zip(rc["energy_runs"], eplus["energy_runs"])):
    rc_kwh = rc_run["annual_delivered_kwh"]
    ep_kwh = ep_run["annual_delivered_kwh"]
    row = {
        "step": step,
        "EnergyPlus kWh/year": ep_kwh,
        "RC kWh/year": rc_kwh,
        "RC above EnergyPlus (%)": 100 * (rc_kwh / ep_kwh - 1),
    }
    rows.append(row)
    print(row)
    assert rc_kwh > 0 and ep_kwh > 0

print("EnergyPlus climate impacts and savings:")
show_table(eplus["report"])
print("RC climate impacts and savings:")
show_table(rc["report"])

The recorded verification run on 14 October 2026 gave EnergyPlus annual energy of 78,303 / 70,257 / 30,511 kWh and RC energy of 92,808 / 83,795 / 34,846 kWh. RC was about 14–19% higher. Both showed about a 10% window improvement and about a 60% improvement after both changes. These are results from this illustrative case, not an allowed-error threshold. The exact recorded values, package versions, weather checksum and assumed climate factors are in results/engine_comparison.json.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = ["Baseline", "Windows", "Windows + insulation"]
for engine_name, result, offset, color in [
    ("EnergyPlus", eplus, -0.18, "#0072B2"),
    ("RC", rc, 0.18, "#E69F00"),
]:
    axes[0].bar(
        [i + offset for i in range(3)],
        [r["annual_delivered_kwh"] for r in result["energy_runs"]],
        0.34,
        label=engine_name,
        color=color,
    )
    axes[1].bar(
        [i + offset for i in range(3)],
        [r["total_kgco2e"] for r in result["report"].table()],
        0.34,
        label=engine_name,
        color=color,
    )
for axis in axes:
    axis.set_xticks(range(3), labels, rotation=15)
    axis.legend()
    axis.grid(axis="y", alpha=0.3)
    axis.set_axisbelow(True)
axes[0].set_ylabel("Annual delivered energy (kWh/year)")
axes[1].set_ylabel("Climate impact over 30 years (kg CO₂e)")
figure.tight_layout()
figure

Material impacts are identical for these two runs. Carrier and removal climate factors are the same explicit assumptions, so the different total savings come from energy differences. All actual engine files and reports remain in their separate run folders.

In [ ]:
import math

for rc_row, ep_row in zip(rc["report"].table(), eplus["report"].table()):
    assert math.isclose(
        rc_row["material_so_far_kgco2e"],
        ep_row["material_so_far_kgco2e"],
        rel_tol=1e-12,
    )
print("RC outputs:", rc["directory"])
print("EnergyPlus outputs:", eplus["directory"])